# TerraPulse: NISAR surface change pipeline

Reading Earth's signals. Revealing its changes.

This notebook turns a year of NISAR L-band radar (GCOV, track 164, frame 76, Lower Sindh, Pakistan) into web layers for the TerraPulse app:

1. **Search** the ASF archive and keep one product per date from a single processing version (PROVISIONAL, CRID P05023), plus the newest VALIDATED date.
2. **Ingest**: stream only the study area out of each 6 GB HDF5 file (parallel range requests of the compressed chunks) and average it onto a common 100 m grid.
3. **Process**: open water, flooded vegetation, water occurrence, change metrics, surface rhythm classes, zone statistics, NASA POWER rainfall.
4. **Export** small PNG layers and a manifest to `web/data/lower-indus`, then download them as a zip.

**How to run in Google Colab:** Runtime > Run all. When asked, paste your NASA Earthdata Login token (from https://urs.earthdata.nasa.gov/profile, Generate Token). The token is read with a hidden prompt and is never saved in this notebook.

In [ ]:
# 1. Get the code
import os, sys, pathlib, zipfile

REPO_URL = ""  # optional: https://github.com/<user>/<repo>.git  (leave empty to upload terrapulse_code.zip)
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    ROOT = pathlib.Path("/content/terrapulse")
    if not (ROOT / "terrapulse").exists():
        if REPO_URL:
            !git clone -q {REPO_URL} {ROOT}
        else:
            from google.colab import files
            print("Upload terrapulse_code.zip (the project folder zipped)")
            up = files.upload()
            name = next(iter(up))
            ROOT.mkdir(parents=True, exist_ok=True)
            with zipfile.ZipFile(name) as z:
                z.extractall(ROOT)
            # accept zips that contain a top-level folder
            inner = [p for p in ROOT.iterdir() if p.is_dir() and (p / "terrapulse").exists()]
            if not (ROOT / "terrapulse").exists() and inner:
                ROOT = inner[0]
    %pip install -q asf_search fsspec aiohttp h5py scipy pillow
else:
    ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()

os.chdir(ROOT)
print("Working in", ROOT)
assert (ROOT / "terrapulse" / "__main__.py").exists(), "terrapulse package not found"

In [ ]:
# 2. Earthdata Login token (hidden input, kept only in this session's memory)
from getpass import getpass
if not os.environ.get("EARTHDATA_TOKEN"):
    os.environ["EARTHDATA_TOKEN"] = getpass("Earthdata Login token: ").strip()
print("token set:", bool(os.environ["EARTHDATA_TOKEN"]))

In [ ]:
# 3. Search the archive and choose one product per date
!{sys.executable} -m terrapulse search

In [ ]:
# 4. Stream the study area from every date onto the 100 m grid (resumable: re-run to continue)
WORKERS = 32 if IN_COLAB else 8
!{sys.executable} -u -m terrapulse ingest --workers {WORKERS}

In [ ]:
# 5. Compute products and write web layers
!{sys.executable} -u -m terrapulse process

In [ ]:
# 6. Quick look
import json
import matplotlib.pyplot as plt
from PIL import Image

web = ROOT / "web" / "data" / "lower-indus"
man = json.loads((web / "manifest.json").read_text())
print("dates:", len(man["dates"]), " thresholds:", man["params"]["water_hh_db"], man["params"]["water_hv_db"])
fig, ax = plt.subplots(1, 3, figsize=(18, 6))
ax[0].imshow(Image.open(web / "products" / "water_occurrence.png"), cmap="Blues"); ax[0].set_title("Water occurrence")
ax[1].imshow(Image.open(web / "products" / "rhythm.png").convert("RGB")); ax[1].set_title("Surface rhythm classes")
d = man["dates"][len(man["dates"]) // 2]["date"]
ax[2].imshow(Image.open(web / "dates" / f"{d}_hh.png"), cmap="gray"); ax[2].set_title(f"HH {d}")
for a in ax: a.axis("off")
plt.show()
z = man["zones"]["all"]["series"]
plt.figure(figsize=(14, 3))
plt.plot([x["date"] for x in man["dates"]], [s["water_km2"] if s else None for s in z], "o-")
plt.xticks(rotation=60); plt.ylabel("open water km2"); plt.title("Open water, whole study area"); plt.show()

In [ ]:
# 7. Package results: web layers (small) and the 100 m cache (lets the laptop re-run processing without downloading NISAR again)
import shutil
shutil.make_archive("terrapulse_web_data", "zip", ROOT / "web" / "data")
shutil.make_archive("terrapulse_cache", "zip", ROOT / "cache")
for f in ["terrapulse_web_data.zip", "terrapulse_cache.zip"]:
    print(f, round(os.path.getsize(f) / 1e6, 1), "MB")
if IN_COLAB:
    from google.colab import files
    files.download("terrapulse_web_data.zip")
    files.download("terrapulse_cache.zip")